# Etapa 5d — Algoritmo final en una función y máscaras de desarrollo

El algoritmo adoptado en `05b` y ampliado en la ronda de mejoras de `05e` (cada filtro también en cada escala por separado, 26 mapas; plan, desvío 14.6) queda en `src/pipeline.py` (`segment_image`; la referencia clásica en `segment_reference`), con un script para imágenes nuevas (`scripts/segmentar.py`). Este cuaderno:

1. **Máscaras de desarrollo con validación cruzada:** cada imagen se segmenta con los pesos y el umbral elegidos sin su persona (las 5 vueltas de siempre) y se guardan en `results/cache/mascaras_cv/`: el algoritmo final y la referencia, cada uno con fracción fija y con valor fijo (desvío 14.7). Las usan las etapas 6 y 7.
2. **Verificación:** Matthews medio de esas máscaras contra el de `05e` (escalas separadas con la fracción, 0,7255) y `05b` (detector de líneas con la fracción 0,7022 y con el valor fijo 0,7040). Tiene que dar lo mismo. La combinación con valor fijo es nueva.
3. **Pesos, fracción y valor fijo con las 30 imágenes de desarrollo**, guardados en `models/pesos_combinacion.json` para el script. Se confirman al congelar (etapa 9).


In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_split
from src.evaluation import map_images
from src.pipeline import choose_fraction, choose_value, fit_weights, save_weights
from src.split import person_folds

ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))

In [2]:
def pass1(image_id):
    from src.pipeline import training_pixels
    return image_id, training_pixels(image_id)

t = time.time()
samples = dict(map_images(pass1, ids, workers=4))
fits = {}
for k in range(5):
    tr = list(folds.index[folds != k])
    w = fit_weights(samples, tr)
    fits[k] = {"w": w, "combinación": (choose_fraction(samples, w, tr), choose_value(samples, w, tr)),
               "referencia": (choose_fraction(samples, None, tr), choose_value(samples, None, tr))}
print(f"Pesos y umbrales por vuelta listos en {time.time() - t:.0f} s")
pd.DataFrame({(m, r): [fits[k][m][j] for k in range(5)] for m in ("combinación", "referencia")
              for j, r in enumerate(("fracción", "valor fijo"))}).round(4)

Pesos y umbrales por vuelta listos en 531 s


combinación            referencia           
     fracción valor fijo   fracción valor fijo
0        0.10     0.8415       0.09     0.9710
1        0.10     0.8434       0.09     0.9761
2        0.09     0.8430       0.09     0.9284
3        0.09     0.8440       0.09     0.9508
4        0.09     0.8334       0.09     0.9451

In [3]:
def pass2(image_id):
    from src.data import load_fov, load_image, load_vessels
    from src.metrics import matthews
    from src.pipeline import compute_features, save_dev_masks, segment_image, segment_reference
    rgb = load_image(image_id)
    f, _ = compute_features(rgb)
    fit = fits[int(folds[image_id])]
    (cf, cv), (rf, rv) = fit["combinación"], fit["referencia"]
    masks = {"combinación": segment_image(rgb, fit["w"], fraction=cf, features=f),
             "referencia": segment_reference(rgb, fraction=rf, features=f),
             "combinación valor fijo": segment_image(rgb, fit["w"], fraction=None, value=cv, features=f),
             "referencia valor fijo": segment_reference(rgb, fraction=None, value=rv, features=f)}
    save_dev_masks(image_id, masks)
    truth, fov = load_vessels(image_id), load_fov(image_id)
    return {"image": image_id, **{k: matthews(m, truth, fov) for k, m in masks.items()}}

t = time.time()
M = pd.DataFrame(map_images(pass2, ids, workers=4)).set_index("image")
print(f"Máscaras guardadas en {time.time() - t:.0f} s")
print(M.mean().round(4).to_string())

Máscaras guardadas en 731 s
combinación               0.7255
referencia                0.7022
combinación valor fijo    0.7304
referencia valor fijo     0.7040


In [4]:
final = fit_weights(samples, ids)
final_fraction, final_value = choose_fraction(samples, final, ids), choose_value(samples, final, ids)
save_weights(final, fraction=final_fraction, value=final_value,
             note="Ajustados con las 30 imágenes de desarrollo (05d, 31/10); se confirman al congelar.")
print(f"Con las 30: fracción {final_fraction} (config.SEGMENT_FRACTION = {C.SEGMENT_FRACTION}), valor fijo {final_value:.4f}")
print(pd.Series(final, index=["ordenada"] + C.SEGMENT_FEATURES).round(3).to_string())

Con las 30: fracción 0.09 (config.SEGMENT_FRACTION = 0.09), valor fijo 0.8257
ordenada                -1.868
verde corregido          0.883
sombrero de copa         0.722
detector de líneas      -1.949
filtro adaptado          0.295
Gabor                    0.809
Frangi                   1.066
detector de líneas 2    -0.104
filtro adaptado 2       -0.030
Gabor 2                 -0.651
Frangi 2                -0.537
detector de líneas 4     0.219
filtro adaptado 4        0.221
Gabor 4                  0.052
Frangi 4                 0.597
detector de líneas 8     0.726
filtro adaptado 8        0.792
Gabor 8                 -1.298
Frangi 8                -1.327
detector de líneas 16    1.175
filtro adaptado 16      -0.370
Gabor 16                 0.044
Frangi 16               -0.687
detector de líneas 24    3.368
filtro adaptado 24      -0.458
Gabor 24                -0.173
Frangi 24               -1.107
